# 03 — M5 Retail Demand Forecasting

**Objective:** Build a manageable item/store demand forecasting experiment from the M5-style wide sales table, calendar, and selling-price files.

## Expected files
Place in `data/m5/`:
- `sales_train_validation.csv`
- `calendar.csv`
- `sell_prices.csv`

**Compute note:** M5 is large. Start with a small number of item/store series. This notebook creates lag features from past sales and uses a chronological validation period. It is a baseline research workflow, not a full reproduction of the official M5 competition evaluation.

In [ ]:
# 1. Imports
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

DATA_DIR = Path("data/m5")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
RANDOM_STATE = 42

In [ ]:
# 2. Load M5 source files
required = ["sales_train_validation.csv", "calendar.csv", "sell_prices.csv"]
missing = [f for f in required if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing {missing} in {DATA_DIR.resolve()}")

sales = pd.read_csv(DATA_DIR / "sales_train_validation.csv")
calendar = pd.read_csv(DATA_DIR / "calendar.csv")
prices = pd.read_csv(DATA_DIR / "sell_prices.csv")

print("Sales:", sales.shape, "Calendar:", calendar.shape, "Prices:", prices.shape)
display(sales.head(2))

In [ ]:
# 3. Select a manageable sample of series and convert wide sales to long format
N_SERIES = 100  # increase only if your RAM/compute permits
series_ids = sales["id"].drop_duplicates().head(N_SERIES)
sales_sample = sales[sales["id"].isin(series_ids)].copy()

id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales_sample.columns if c.startswith("d_")]
long_df = sales_sample.melt(
    id_vars=[c for c in id_cols if c in sales_sample.columns],
    value_vars=day_cols, var_name="d", value_name="sales"
)
long_df = long_df.merge(calendar, on="d", how="left")
long_df["date"] = pd.to_datetime(long_df["date"], errors="coerce")
join_cols = ["store_id", "item_id", "wm_yr_wk"]
long_df = long_df.merge(prices[join_cols + ["sell_price"]], on=join_cols, how="left")
long_df["sales"] = pd.to_numeric(long_df["sales"], errors="coerce").fillna(0)
long_df = long_df.sort_values(["id", "date"]).reset_index(drop=True)

print("Long sample:", long_df.shape)
display(long_df.head())

In [ ]:
# 4. Feature engineering — all lag values are shifted, so current/future target is not used
long_df["day_of_week"] = long_df["date"].dt.dayofweek
long_df["month"] = long_df["date"].dt.month
long_df["year"] = long_df["date"].dt.year
long_df["week_of_year"] = long_df["date"].dt.isocalendar().week.astype(int)

for lag in [7, 28]:
    long_df[f"lag_{lag}"] = long_df.groupby("id")["sales"].shift(lag)

long_df["rolling_mean_7"] = long_df.groupby("id")["sales"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=7).mean()
)
long_df["rolling_mean_28"] = long_df.groupby("id")["sales"].transform(
    lambda s: s.shift(1).rolling(28, min_periods=28).mean()
)

feature_cols = [
    "day_of_week", "month", "year", "week_of_year",
    "sell_price", "lag_7", "lag_28", "rolling_mean_7", "rolling_mean_28"
]
model_df = long_df.dropna(subset=feature_cols + ["sales", "date"]).copy()
print("Modeling rows:", model_df.shape)

In [ ]:
# 5. Chronological split
unique_dates = np.sort(model_df["date"].unique())
split_date = unique_dates[int(len(unique_dates) * 0.8)]
train = model_df[model_df["date"] < split_date]
valid = model_df[model_df["date"] >= split_date]

X_train, y_train = train[feature_cols], train["sales"]
X_valid, y_valid = valid[feature_cols], valid["sales"]
print("Validation starts:", pd.Timestamp(split_date))
print("Train rows:", len(train), "Validation rows:", len(valid))

In [ ]:
# 6. Train XGBoost baseline
model = XGBRegressor(
    n_estimators=400, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", random_state=RANDOM_STATE,
    n_jobs=-1
)
model.fit(X_train, y_train)
pred = np.maximum(model.predict(X_valid), 0)

metrics = {
    "MAE": mean_absolute_error(y_valid, pred),
    "RMSE": mean_squared_error(y_valid, pred) ** 0.5
}
print("XGBoost validation metrics:", metrics)
m5_results = pd.DataFrame([
    {"Dataset": "M5", "Model": "XGBoost", "Metric": k, "Value": float(v)}
    for k, v in metrics.items()
])
display(m5_results)
m5_results.to_csv(RESULTS_DIR / "m5_model_results.csv", index=False)

In [ ]:
# 7. Aggregate validation predictions by date for an overview plot
plot_df = valid[["date", "sales"]].copy()
plot_df["prediction"] = pred
daily = plot_df.groupby("date")[["sales", "prediction"]].sum()

plt.figure(figsize=(13, 4))
plt.plot(daily.index, daily["sales"], label="Actual")
plt.plot(daily.index, daily["prediction"], label="Predicted")
plt.title("M5 Sample — Aggregate Validation Demand")
plt.xlabel("Date"); plt.ylabel("Units sold"); plt.legend(); plt.grid(alpha=.2); plt.show()

## Optional extension: LightGBM and LSTM

This starter notebook establishes an XGBoost baseline. To make a fair three-model comparison, implement LightGBM and an LSTM using the **same series sample, forecast horizon, and chronological validation dates**. Do not compare results from different data splits or horizons. Record sampling and compute settings in your report.

## Limitations and reporting

- This notebook samples the first `N_SERIES` series for a manageable experiment; it is not the full M5 competition benchmark.
- A single global chronological split is used. Consider rolling-origin backtesting for stronger evidence.
- The feature set is intentionally compact. Add event, SNAP, and category/store features only after validating their availability at forecast time.
- For a true multi-step forecast, ensure all lag features are generated recursively or from information available at the forecast origin.